## Maintenance notice

> **Historical experiment record.** The outputs in this notebook were produced before the negative-sampling correction. Use the versioned modules under src/sales_forecasting/ for the maintained implementation and rerun training before reporting a corrected score.

The generated cells and outputs are retained for experiment traceability; current fixes, tests, portable paths, and evaluation documentation live in the repository source and docs.

# Sales Forecasting Pipeline

This notebook uses `%%writefile` to reconstruct the training pipeline exactly as it was in the original repository. By doing this, we avoid any namespace conflicts (like overwritten `main()` functions) and ensure the execution works exactly as running the scripts in the terminal.

### `project_paths.py`

In [1]:
%%writefile project_paths.py
from pathlib import Path

# Cấu hình đường dẫn riêng cho môi trường Kaggle
ROOT_DIR = Path("/kaggle/working")
RAW_DATA_DIR = Path("/kaggle/input/datasets/thnhhuytrn/saleforecasting")
OUTPUT_DIR = ROOT_DIR  # Lưu trực tiếp vào /kaggle/working/ để Kaggle nhận diện file output dễ dàng

# Đường dẫn 3 file parquet
TRANSACTIONS_PATH = RAW_DATA_DIR / "transaction_full_2025.parquet"
ITEMS_PATH = RAW_DATA_DIR / "items.parquet"
EVENT_PATH = RAW_DATA_DIR / "event_full_2025.parquet"

CACHE_DIR = OUTPUT_DIR / "cache"
LOG_DIR = OUTPUT_DIR / "logs"

def ensure_output_dirs():
    for path in [CACHE_DIR, LOG_DIR, OUTPUT_DIR / "eda"]:
        path.mkdir(parents=True, exist_ok=True)

def first_existing(*paths):
    for path in paths:
        candidate = Path(path)
        if candidate.exists():
            return candidate
    return Path(paths[0])

Writing project_paths.py


### `train_predict.py`

In [2]:
%%writefile train_predict.py
import argparse
import calendar
import json
import pickle
import time
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
from catboost import CatBoostClassifier, CatBoostRegressor, Pool

from project_paths import CACHE_DIR, OUTPUT_DIR, RAW_DATA_DIR, ensure_output_dirs


TRANSACTIONS_PATH = RAW_DATA_DIR / "transaction_full_2025.parquet"
ITEMS_PATH = RAW_DATA_DIR / "items.parquet"
DAILY_PATH = CACHE_DIR / "daily_quantity.parquet"
SUBMISSION_PATH = OUTPUT_DIR / "submission.pkl"
MODEL_DIR = OUTPUT_DIR / "models"

BAD_DAY_QTY_THRESHOLD = 100_000
KNOWN_BAD_DAYS = [
    date(2025, 1, 29),
    date(2025, 1, 30),
    date(2025, 1, 31),
    date(2025, 7, 4),
    date(2025, 7, 18),
    date(2025, 9, 29),
    date(2025, 10, 27),
]
RECENT_WINDOWS = (30, 60, 90, 180, 365)
CAT_COLUMNS = [
    "location",
    "item_id",
    "category_l1",
    "category_l2",
    "category_l3",
    "category",
    "brand",
    "manufacturer",
    "size",
]
ITEM_META_COLUMNS = ["category_l1", "category_l2", "category_l3", "category", "brand", "manufacturer"]


def log(message: str) -> None:
    print(message, flush=True)


def step_timer(name: str):
    class Timer:
        def __enter__(self):
            self.start = time.perf_counter()
            log(f"START {name}")

        def __exit__(self, exc_type, exc, traceback):
            elapsed = time.perf_counter() - self.start
            status = "FAILED" if exc_type else "DONE"
            log(f"{status} {name} in {elapsed:.1f}s")

    return Timer()


def month_bounds(year: int, month: int) -> tuple[date, date]:
    return date(year, month, 1), date(year, month, calendar.monthrange(year, month)[1])


def ensure_daily_cache() -> None:
    if DAILY_PATH.exists():
        return
    log(f"Missing {DAILY_PATH}; building daily cache from raw transactions")
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    (
        pl.scan_parquet(TRANSACTIONS_PATH)
        .filter(pl.col("event_type") == "Purchase")
        .select(
            "location",
            pl.col("item_id").cast(pl.Utf8).alias("item_id"),
            pl.col("updated_date").dt.date().alias("date"),
            pl.col("quantity").cast(pl.Float32).alias("quantity"),
        )
        .group_by(["location", "item_id", "date"])
        .agg(pl.col("quantity").sum().alias("y"))
        .sink_parquet(DAILY_PATH)
    )


def validate_inputs(args) -> None:
    missing = [path for path in [TRANSACTIONS_PATH, ITEMS_PATH] if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Missing required files: {missing}")
    ensure_daily_cache()
    if args.task_type == "GPU":
        log("Running CatBoost GPU preflight")
        try:
            toy_x = pd.DataFrame({"x": [0, 1, 2, 3], "cat": ["a", "b", "a", "b"]})
            toy_y = np.array([0, 1, 0, 1])
            toy_model = CatBoostClassifier(
                iterations=2,
                depth=2,
                learning_rate=0.1,
                loss_function="Logloss",
                task_type="GPU",
                devices=args.devices,
                verbose=False,
            )
            toy_model.fit(Pool(toy_x, toy_y, cat_features=["cat"]))
        except Exception as exc:
            raise RuntimeError(
                "CatBoost GPU preflight failed before expensive training. "
                "Switch to --task-type CPU or fix the Colab CUDA/CatBoost environment."
            ) from exc


def detect_bad_days(mode: str) -> list[date]:
    if mode == "none":
        return []
    if mode == "known":
        return KNOWN_BAD_DAYS
    daily_totals = (
        pl.scan_parquet(DAILY_PATH)
        .group_by("date")
        .agg(pl.col("y").sum().alias("qty"))
        .collect()
    )
    calendar_df = pl.DataFrame(
        {"date": pl.date_range(date(2025, 1, 1), date(2025, 12, 31), interval="1d", eager=True)}
    )
    full = calendar_df.join(daily_totals, on="date", how="left").with_columns(pl.col("qty").fill_null(0))
    bad_days = sorted(full.filter(pl.col("qty") < BAD_DAY_QTY_THRESHOLD)["date"].to_list())
    if len(bad_days) > 30:
        log(
            f"Threshold anomaly detection found {len(bad_days)} bad days, "
            "likely because this is a subset. Falling back to known full-data anomaly dates."
        )
        return KNOWN_BAD_DAYS
    return bad_days


def valid_days_between(start: date, end: date, bad_days: set[date]) -> int:
    days = 0
    current = start
    while current <= end:
        if current not in bad_days:
            days += 1
        current += timedelta(days=1)
    return days


def candidate_locations() -> pl.DataFrame:
    return (
        pl.scan_parquet(TRANSACTIONS_PATH)
        .select(pl.col("location").cast(pl.Utf8).alias("location"))
        .unique()
        .sort("location")
        .collect()
    )


def active_items_with_metadata() -> pl.DataFrame:
    text_defaults = {
        "category_l1": "unknown",
        "category_l2": "unknown",
        "category_l3": "unknown",
        "category": "unknown",
        "brand": "unknown",
        "manufacturer": "unknown",
        "size": "unknown",
    }
    return (
        pl.scan_parquet(ITEMS_PATH)
        .filter(pl.col("sale_status") == 1)
        .select(
            pl.col("item_id").cast(pl.Utf8),
            pl.col("price").cast(pl.Float64).fill_null(0).alias("price"),
            *[
                pl.col(col).cast(pl.Utf8).fill_null(default).alias(col)
                for col, default in text_defaults.items()
            ],
        )
        .unique("item_id")
        .sort("item_id")
        .collect()
    )


def daily_valid_lf(bad_days: set[date]) -> pl.LazyFrame:
    return (
        pl.scan_parquet(DAILY_PATH)
        .select(
            pl.col("location").cast(pl.Utf8).alias("location"),
            pl.col("item_id").cast(pl.Utf8).alias("item_id"),
            pl.col("date").cast(pl.Date),
            pl.col("y").cast(pl.Float64).alias("quantity"),
        )
        .filter(~pl.col("date").is_in(list(bad_days)))
    )


def monthly_target(
    daily_lf: pl.LazyFrame,
    active_items: pl.DataFrame,
    target_year: int,
    target_month: int,
    bad_days: set[date],
) -> pl.DataFrame:
    start, end = month_bounds(target_year, target_month)
    valid_days = valid_days_between(start, end, bad_days)
    calendar_days = (end - start).days + 1
    return (
        daily_lf.filter((pl.col("date") >= start) & (pl.col("date") <= end))
        .group_by(["location", "item_id"])
        .agg(pl.col("quantity").sum().alias("quantity"))
        .join(active_items.lazy().select("item_id"), on="item_id", how="inner")
        .with_columns((pl.col("quantity") * calendar_days / valid_days).alias("quantity"))
        .collect()
    )


def sample_frame(df: pl.DataFrame, max_rows: int, seed: int) -> pl.DataFrame:
    if df.height <= max_rows:
        return df
    return df.sample(n=max_rows, seed=seed, shuffle=True)


def negative_sample(
    locations: pl.DataFrame,
    active_items: pl.DataFrame,
    positives: pl.DataFrame,
    n_rows: int,
    seed: int,
) -> pl.DataFrame:
    # Oversample candidate pairs, then anti-join positives. This avoids collecting
    # the full 7.1M-pair grid for every training month.
    rng = np.random.default_rng(seed)
    loc_pd = locations.to_pandas()
    item_pd = active_items.select("item_id").to_pandas()
    chunks: list[pl.DataFrame] = []
    positive_keys = positives.select(["location", "item_id"]).unique()
    remaining = n_rows
    while remaining > 0:
        draw = max(remaining * 3, 50_000)
        sampled_locations = loc_pd.iloc[rng.integers(0, len(loc_pd), size=draw)].reset_index(drop=True)
        sampled_items = item_pd.iloc[rng.integers(0, len(item_pd), size=draw)].reset_index(drop=True)
        candidates = pl.from_pandas(
            pd.DataFrame(
                {
                    "location": sampled_locations["location"].astype(str),
                    "item_id": sampled_items["item_id"].astype(str),
                }
            )
        ).unique()
        negatives = candidates.join(positive_keys, on=["location", "item_id"], how="anti")
        if negatives.height:
            take = min(remaining, negatives.height)
            chunks.append(negatives.head(take))
            remaining -= take
    return pl.concat(chunks).unique().head(n_rows).with_columns(pl.lit(0.0).alias("quantity"))


def aggregate_features(daily_lf: pl.LazyFrame, cutoff: date, prefix: str, keys: list[str]) -> pl.DataFrame:
    lf = daily_lf.filter(pl.col("date") <= cutoff)
    aggs = [
        pl.col("quantity").sum().alias(f"{prefix}_qty_all"),
        pl.col("date").n_unique().alias(f"{prefix}_active_days_all"),
        pl.col("date").max().alias(f"{prefix}_last_sale_date"),
    ]
    for window in RECENT_WINDOWS:
        start = cutoff - timedelta(days=window - 1)
        in_window = pl.col("date") >= start
        aggs.extend(
            [
                pl.col("quantity").filter(in_window).sum().alias(f"{prefix}_qty_{window}d"),
                pl.col("date").filter(in_window).n_unique().alias(f"{prefix}_active_days_{window}d"),
            ]
        )
    return lf.group_by(keys).agg(aggs).collect()


def aggregate_meta_features(
    daily_lf: pl.LazyFrame,
    active_items: pl.DataFrame,
    cutoff: date,
    prefix: str,
    keys: list[str],
) -> pl.DataFrame:
    meta_lf = active_items.lazy().select(["item_id", *ITEM_META_COLUMNS])
    return aggregate_features(daily_lf.join(meta_lf, on="item_id", how="left"), cutoff, prefix, keys)


def build_feature_tables(daily_lf: pl.LazyFrame, active_items: pl.DataFrame, cutoff: date) -> dict[str, pl.DataFrame]:
    with step_timer(f"feature tables cutoff={cutoff}"):
        return {
            "pair": aggregate_features(daily_lf, cutoff, "pair", ["location", "item_id"]),
            "item": aggregate_features(daily_lf, cutoff, "item", ["item_id"]),
            "location": aggregate_features(daily_lf, cutoff, "location", ["location"]),
            "category_l1": aggregate_meta_features(daily_lf, active_items, cutoff, "cat1", ["category_l1"]),
            "category_l2": aggregate_meta_features(daily_lf, active_items, cutoff, "cat2", ["category_l2"]),
            "brand": aggregate_meta_features(daily_lf, active_items, cutoff, "brand", ["brand"]),
            "location_category_l1": aggregate_meta_features(
                daily_lf, active_items, cutoff, "loc_cat1", ["location", "category_l1"]
            ),
        }


def add_features_from_tables(
    pairs: pl.DataFrame,
    active_items: pl.DataFrame,
    feature_tables: dict[str, pl.DataFrame],
    cutoff: date,
    target_month: int,
    target_days: int,
) -> pd.DataFrame:
    features = (
        pairs.select(["location", "item_id", "quantity"] if "quantity" in pairs.columns else ["location", "item_id"])
        .join(active_items, on="item_id", how="left")
        .join(feature_tables["pair"], on=["location", "item_id"], how="left")
        .join(feature_tables["item"], on="item_id", how="left")
        .join(feature_tables["location"], on="location", how="left")
        .join(feature_tables["category_l1"], on="category_l1", how="left")
        .join(feature_tables["category_l2"], on="category_l2", how="left")
        .join(feature_tables["brand"], on="brand", how="left")
        .join(feature_tables["location_category_l1"], on=["location", "category_l1"], how="left")
        .with_columns(
            pl.lit(target_month).alias("target_month"),
            pl.lit(target_days).alias("target_days"),
        )
    )

    date_columns = [col for col in features.columns if col.endswith("_last_sale_date")]
    for col in date_columns:
        features = features.with_columns(
            (pl.lit(cutoff) - pl.col(col)).dt.total_days().fill_null(999).alias(col.replace("_last_sale_date", "_days_since_sale"))
        ).drop(col)

    numeric_columns = [col for col, dtype in zip(features.columns, features.dtypes) if dtype.is_numeric()]
    features = features.with_columns(pl.col(numeric_columns).fill_null(0))
    features = features.with_columns(
        (pl.col("pair_qty_30d") / (pl.col("pair_qty_90d") + 1)).alias("pair_recent_share_30_90"),
        (pl.col("item_qty_30d") / (pl.col("item_qty_90d") + 1)).alias("item_recent_share_30_90"),
        (pl.col("location_qty_30d") / (pl.col("location_qty_90d") + 1)).alias("location_recent_share_30_90"),
        (pl.col("pair_qty_90d") / (pl.col("item_qty_90d") + 1)).alias("pair_item_share_90d"),
        (pl.col("pair_qty_90d") / (pl.col("location_qty_90d") + 1)).alias("pair_location_share_90d"),
    )
    for col in CAT_COLUMNS:
        if col in features.columns:
            features = features.with_columns(pl.col(col).cast(pl.Utf8).fill_null("unknown"))
    return features.to_pandas()


def build_training_data(args, bad_days: set[date], locations: pl.DataFrame, active_items: pl.DataFrame):
    daily_lf = daily_valid_lf(bad_days)
    frames: list[pd.DataFrame] = []
    months = [month for month in range(args.start_month, args.end_month + 1)]
    for month in months:
        cutoff = month_bounds(2025, month)[0] - timedelta(days=1)
        _, target_end = month_bounds(2025, month)
        target_days = calendar.monthrange(2025, month)[1]
        target = monthly_target(daily_lf, active_items, 2025, month, bad_days)
        positives = sample_frame(target.filter(pl.col("quantity") > 0), args.max_positive_per_month, seed=10_000 + month)
        negatives = negative_sample(
            locations,
            active_items,
            positives,
            min(args.max_negative_per_month, max(positives.height * args.negative_ratio, 1)),
            seed=20_000 + month,
        )
        month_pairs = pl.concat([positives, negatives], how="vertical")
        log(
            f"Month {month:02d}: cutoff={cutoff}, positives={positives.height:,}, "
            f"negatives={negatives.height:,}, target_end={target_end}"
        )
        feature_tables = build_feature_tables(daily_lf, active_items, cutoff)
        month_features = add_features_from_tables(month_pairs, active_items, feature_tables, cutoff, month, target_days)
        frames.append(month_features)
    return pd.concat(frames, ignore_index=True)


def score_metric(actual: np.ndarray, pred: np.ndarray) -> float:
    mask = actual != 0
    if not mask.any():
        return 0.0
    ape = np.abs(actual[mask] - pred[mask]) / np.abs(actual[mask])
    return round(100.0 - float(ape.mean()) * 100.0, 4)


def mape_sample_weight(y: np.ndarray, args) -> np.ndarray:
    if args.mape_weight_power <= 0:
        return np.ones_like(y, dtype=np.float64)
    clipped = np.clip(y, args.mape_weight_floor, None)
    weights = 1.0 / np.power(clipped, args.mape_weight_power)
    return weights / weights.mean()


def quantity_bucket_labels(y: np.ndarray) -> np.ndarray:
    rounded = np.rint(np.clip(y, 1, None)).astype(np.int64)
    labels = np.zeros_like(rounded)
    labels[rounded == 1] = 0
    labels[rounded == 2] = 1
    labels[rounded == 3] = 2
    labels[(rounded >= 4) & (rounded <= 5)] = 3
    labels[(rounded >= 6) & (rounded <= 10)] = 4
    labels[(rounded >= 11) & (rounded <= 20)] = 5
    labels[(rounded >= 21) & (rounded <= 50)] = 6
    labels[rounded >= 51] = 7
    return labels


def bucket_values_from_training(y: np.ndarray, labels: np.ndarray) -> np.ndarray:
    fallback = np.array([1, 2, 3, 4.5, 8, 15, 30, 60], dtype=np.float64)
    values = fallback.copy()
    for label in range(len(values)):
        bucket_y = y[labels == label]
        if len(bucket_y):
            values[label] = float(np.median(bucket_y))
    return values


def train_models(train_df: pd.DataFrame, args):
    y = train_df.pop("quantity").astype(float).to_numpy()
    positive = (y > 0).astype(int)
    cat_features = [col for col in CAT_COLUMNS if col in train_df.columns]

    clf = CatBoostClassifier(
        iterations=args.classifier_iterations,
        learning_rate=args.learning_rate,
        depth=args.depth,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=args.seed,
        thread_count=args.thread_count,
        task_type=args.task_type,
        devices=args.devices,
        verbose=100,
    )
    with step_timer("train positive classifier"):
        class_weights = np.where(positive == 1, args.classifier_positive_weight, 1.0)
        clf.fit(Pool(train_df, positive, cat_features=cat_features, weight=class_weights))

    reg = CatBoostRegressor(
        iterations=args.regressor_iterations,
        learning_rate=args.learning_rate,
        depth=args.depth,
        loss_function=args.regressor_loss,
        random_seed=args.seed,
        thread_count=args.thread_count,
        task_type=args.task_type,
        devices=args.devices,
        verbose=100,
    )
    positive_df = train_df.loc[positive == 1].copy()
    positive_y = y[positive == 1]
    if args.regressor_target == "log":
        reg_target = np.log1p(positive_y)
    else:
        reg_target = positive_y
    reg_weights = mape_sample_weight(positive_y, args)
    with step_timer("train positive quantity regressor"):
        reg.fit(Pool(positive_df, reg_target, cat_features=cat_features, weight=reg_weights))

    bucket_model = None
    bucket_values = None
    if args.use_bucket_model:
        bucket_labels = quantity_bucket_labels(positive_y)
        bucket_values = bucket_values_from_training(positive_y, bucket_labels)
        if len(np.unique(bucket_labels)) >= 2:
            bucket_model = CatBoostClassifier(
                iterations=args.bucket_iterations,
                learning_rate=args.learning_rate,
                depth=args.depth,
                loss_function="MultiClass",
                random_seed=args.seed,
                thread_count=args.thread_count,
                task_type=args.task_type,
                devices=args.devices,
                verbose=100,
            )
            with step_timer("train positive quantity bucket classifier"):
                bucket_model.fit(Pool(positive_df, bucket_labels, cat_features=cat_features, weight=reg_weights))
        else:
            log("Skipping bucket classifier: only one positive quantity bucket in training data")

    return clf, reg, bucket_model, bucket_values, list(train_df.columns), cat_features


def predict_raw_components(
    args,
    clf,
    reg,
    bucket_model,
    bucket_values,
    features: pd.DataFrame,
    feature_columns: list[str],
    cat_features: list[str],
) -> tuple[np.ndarray, np.ndarray, np.ndarray | None]:
    x = features[feature_columns]
    pool = Pool(x, cat_features=cat_features)
    proba = clf.predict_proba(pool)[:, 1]
    reg_pred = reg.predict(pool)
    amount = np.expm1(reg_pred) if args.regressor_target == "log" else reg_pred
    amount = np.clip(amount, 0, None)
    bucket_amount = None
    if bucket_model is not None and bucket_values is not None:
        bucket_proba = bucket_model.predict_proba(pool)
        class_count = min(bucket_proba.shape[1], len(bucket_values))
        bucket_amount = bucket_proba[:, :class_count] @ bucket_values[:class_count]
    return proba, amount, bucket_amount


def predict_quantity(
    args,
    clf,
    reg,
    bucket_model,
    bucket_values,
    features: pd.DataFrame,
    feature_columns: list[str],
    cat_features: list[str],
    threshold: float,
    scale: float,
    blend: float,
    bucket_blend: float,
    proba_power: float,
) -> np.ndarray:
    proba, amount, bucket_amount = predict_raw_components(
        args, clf, reg, bucket_model, bucket_values, features, feature_columns, cat_features
    )
    if bucket_amount is not None and bucket_blend > 0:
        amount = (1.0 - bucket_blend) * amount + bucket_blend * bucket_amount
    baseline = baseline_predictions(features)["pair_last_90d_daily"].astype(float)
    proba_factor = np.power(np.clip(proba, 0, 1), proba_power)
    pred = proba_factor * amount * scale
    if blend > 0:
        pred = (1.0 - blend) * pred + blend * baseline
    if threshold > 0:
        pred[proba < threshold] = 0
    pred = np.rint(np.clip(pred, 0, None)).astype(np.int64)
    return pred


def apply_baseline_strategy(features: pd.DataFrame, strategy: str, scale: float) -> np.ndarray:
    baselines = baseline_predictions(features)
    if strategy not in baselines:
        raise ValueError(f"Unknown baseline strategy {strategy}. Options: {sorted(baselines)}")
    pred = baselines[strategy].astype(float) * scale
    return np.rint(np.clip(pred, 0, None)).astype(np.int64)


def baseline_predictions(features: pd.DataFrame) -> dict[str, np.ndarray]:
    target_days = features["target_days"].to_numpy()
    pair_last_30 = np.rint(np.clip(features["pair_qty_30d"].to_numpy(), 0, None)).astype(np.int64)
    pair_last_90_daily = features["pair_qty_90d"].to_numpy() / 90.0
    pair_last_90 = np.rint(np.clip(pair_last_90_daily * target_days, 0, None)).astype(np.int64)
    pair_last_180_daily = features["pair_qty_180d"].to_numpy() / 180.0
    pair_last_180 = np.rint(np.clip(pair_last_180_daily * target_days, 0, None)).astype(np.int64)
    pair_blend = (
        0.50 * features["pair_qty_30d"].to_numpy()
        + 0.30 * pair_last_90_daily * target_days
        + 0.20 * pair_last_180_daily * target_days
    )
    item_share = features["pair_item_share_90d"].to_numpy()
    item_scaled = features["item_qty_30d"].to_numpy() * item_share
    item_share_pred = np.rint(np.clip(item_scaled, 0, None)).astype(np.int64)
    item_share_smooth = (
        (features["pair_qty_90d"].to_numpy() + 0.03 * features["loc_cat1_qty_90d"].to_numpy())
        / (features["item_qty_90d"].to_numpy() + 0.03 * features["cat1_qty_90d"].to_numpy() + 1.0)
    )
    item_smooth_pred = features["item_qty_30d"].to_numpy() * item_share_smooth
    conservative = np.minimum(pair_blend, item_smooth_pred)
    return {
        "pair_last_30d": pair_last_30,
        "pair_last_90d_daily": pair_last_90,
        "pair_last_180d_daily": pair_last_180,
        "pair_blend_recent": np.rint(np.clip(pair_blend, 0, None)).astype(np.int64),
        "item_share_90d": item_share_pred,
        "item_share_smooth": np.rint(np.clip(item_smooth_pred, 0, None)).astype(np.int64),
        "conservative_min": np.rint(np.clip(conservative, 0, None)).astype(np.int64),
    }


def validation_features_for_month(
    args,
    bad_days: set[date],
    active_items: pl.DataFrame,
    month: int,
    positive_only: bool,
) -> pd.DataFrame:
    daily_lf = daily_valid_lf(bad_days)
    cutoff = month_bounds(2025, month)[0] - timedelta(days=1)
    target_days = calendar.monthrange(2025, month)[1]
    target = monthly_target(daily_lf, active_items, 2025, month, bad_days)
    validation_pairs = target.filter(pl.col("quantity") > 0) if positive_only else target
    validation_pairs = sample_frame(validation_pairs, args.validation_positive_rows, seed=30_000 + month)
    feature_tables = build_feature_tables(daily_lf, active_items, cutoff)
    return add_features_from_tables(validation_pairs, active_items, feature_tables, cutoff, month, target_days)


def tune_on_validation(args, bad_days: set[date], active_items: pl.DataFrame, clf, reg, bucket_model, bucket_values, feature_columns, cat_features):
    validation_months = args.validation_months or [args.validation_month]
    validation_frames = [
        validation_features_for_month(args, bad_days, active_items, month, positive_only=True)
        for month in validation_months
    ]
    features = pd.concat(validation_frames, ignore_index=True)
    actual = features["quantity"].astype(float).to_numpy()
    features_no_y = features.drop(columns=["quantity"])

    best = {"score": -1e18, "threshold": 0.0, "scale": 1.0, "blend": 0.0, "bucket_blend": 0.0, "proba_power": 1.0}
    report_rows = []
    for baseline_name in baseline_predictions(features_no_y):
        for baseline_scale in args.baseline_scale_grid:
            pred = apply_baseline_strategy(features_no_y, baseline_name, baseline_scale)
            score = score_metric(actual, pred)
            report_rows.append(
                {
                    "model": baseline_name,
                    "months": ",".join(str(month) for month in validation_months),
                    "threshold": None,
                    "scale": baseline_scale,
                    "blend": None,
                    "bucket_blend": None,
                    "proba_power": None,
                    "score": score,
                    "mean_pred": float(pred.mean()),
                    "nonzero_pred_rate": float((pred > 0).mean()),
                }
            )

    for threshold in args.threshold_grid:
        for scale in args.scale_grid:
            for blend in args.blend_grid:
                for bucket_blend in args.bucket_blend_grid:
                    for proba_power in args.proba_power_grid:
                        pred = predict_quantity(
                            args,
                            clf,
                            reg,
                            bucket_model,
                            bucket_values,
                            features_no_y,
                            feature_columns,
                            cat_features,
                            threshold,
                            scale,
                            blend,
                            bucket_blend,
                            proba_power,
                        )
                        score = score_metric(actual, pred)
                        report_rows.append(
                            {
                                "model": "catboost_two_stage",
                                "months": ",".join(str(month) for month in validation_months),
                                "threshold": threshold,
                                "scale": scale,
                                "blend": blend,
                                "bucket_blend": bucket_blend,
                                "proba_power": proba_power,
                                "score": score,
                                "mean_pred": float(pred.mean()),
                                "nonzero_pred_rate": float((pred > 0).mean()),
                            }
                        )
                        if score > best["score"]:
                            best = {
                                "score": score,
                                "threshold": threshold,
                                "scale": scale,
                                "blend": blend,
                                "bucket_blend": bucket_blend,
                                "proba_power": proba_power,
                            }

    report = pd.DataFrame(report_rows).sort_values("score", ascending=False)
    report_path = MODEL_DIR / "validation_report.csv"
    report.to_csv(report_path, index=False)
    log(f"Validation months {validation_months} best: {best}")
    log(f"Wrote {report_path}")
    log("Top validation rows:")
    log(report.head(8).to_string(index=False))
    return best


def final_prediction(
    args,
    bad_days: set[date],
    locations: pl.DataFrame,
    active_items: pl.DataFrame,
    clf,
    reg,
    bucket_model,
    bucket_values,
    feature_columns,
    cat_features,
    threshold: float,
    scale: float,
    blend: float,
    bucket_blend: float,
    proba_power: float,
):
    daily_lf = daily_valid_lf(bad_days)
    cutoff = date(2025, 12, 31)
    feature_tables = build_feature_tables(daily_lf, active_items, cutoff)
    rows: list[pd.DataFrame] = []
    item_grid = active_items.select("item_id")
    loc_values = locations["location"].to_list()
    if args.max_prediction_locations:
        loc_values = loc_values[: args.max_prediction_locations]
    for start in range(0, len(loc_values), args.location_batch_size):
        batch_locations = pl.DataFrame({"location": loc_values[start : start + args.location_batch_size]})
        pairs = batch_locations.join(item_grid, how="cross")
        features = add_features_from_tables(pairs, active_items, feature_tables, cutoff, 1, 31)
        if args.prediction_strategy == "catboost":
            pred = predict_quantity(
                args,
                clf,
                reg,
                bucket_model,
                bucket_values,
                features,
                feature_columns,
                cat_features,
                threshold,
                scale,
                blend,
                bucket_blend,
                proba_power,
            )
        else:
            pred = apply_baseline_strategy(features, args.prediction_strategy, args.final_scale)
        out = features[["location", "item_id"]].copy()
        out["quantity"] = pred.astype(np.int64)
        rows.append(out)
        log(f"Predicted locations {start + 1:,}-{min(start + args.location_batch_size, len(loc_values)):,} / {len(loc_values):,}")

    submission = pd.concat(rows, ignore_index=True)
    submission = submission[["location", "item_id", "quantity"]]
    submission = pd.DataFrame(
        {
            "location": submission["location"].astype(str).to_numpy(dtype=object),
            "item_id": submission["item_id"].astype(str).to_numpy(dtype=object),
            "quantity": submission["quantity"].astype(np.int64).to_numpy(),
        }
    )
    with SUBMISSION_PATH.open("wb") as f:
        pickle.dump(submission, f)
    return submission


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--mode", choices=["full", "final-only", "validate-only"], default="full")
    parser.add_argument("--bad-day-mode", choices=["threshold", "known", "none"], default="threshold")
    parser.add_argument("--start-month", type=int, default=4)
    parser.add_argument("--end-month", type=int, default=12)
    parser.add_argument("--validation-month", type=int, default=12)
    parser.add_argument("--validation-months", type=int, nargs="+", default=[12])
    parser.add_argument("--max-positive-per-month", type=int, default=120_000)
    parser.add_argument("--max-negative-per-month", type=int, default=240_000)
    parser.add_argument("--negative-ratio", type=int, default=2)
    parser.add_argument("--validation-positive-rows", type=int, default=300_000)
    parser.add_argument("--classifier-iterations", type=int, default=700)
    parser.add_argument("--regressor-iterations", type=int, default=900)
    parser.add_argument("--learning-rate", type=float, default=0.06)
    parser.add_argument("--depth", type=int, default=8)
    parser.add_argument("--regressor-loss", default="RMSE")
    parser.add_argument("--regressor-target", choices=["log", "raw"], default="log")
    parser.add_argument("--use-bucket-model", action=argparse.BooleanOptionalAction, default=True)
    parser.add_argument("--bucket-iterations", type=int, default=500)
    parser.add_argument("--mape-weight-power", type=float, default=0.0)
    parser.add_argument("--mape-weight-floor", type=float, default=1.0)
    parser.add_argument("--classifier-positive-weight", type=float, default=1.0)
    parser.add_argument("--thread-count", type=int, default=-1)
    parser.add_argument("--task-type", choices=["CPU", "GPU"], default="CPU")
    parser.add_argument("--devices", default="0")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--location-batch-size", type=int, default=40)
    parser.add_argument("--max-prediction-locations", type=int, default=0)
    parser.add_argument("--threshold-grid", type=float, nargs="+", default=[0.0, 0.01, 0.02, 0.05, 0.08, 0.12, 0.18, 0.25, 0.35])
    parser.add_argument("--scale-grid", type=float, nargs="+", default=[0.35, 0.45, 0.55, 0.65, 0.75, 0.85, 0.95, 1.1, 1.25])
    parser.add_argument("--blend-grid", type=float, nargs="+", default=[0.0, 0.1, 0.2, 0.35, 0.5])
    parser.add_argument("--bucket-blend-grid", type=float, nargs="+", default=[0.0, 0.25, 0.5, 0.75, 1.0])
    parser.add_argument("--proba-power-grid", type=float, nargs="+", default=[0.0, 0.25, 0.5, 1.0])
    parser.add_argument("--final-threshold", type=float, default=0.02)
    parser.add_argument("--final-scale", type=float, default=1.0)
    parser.add_argument("--final-blend", type=float, default=0.0)
    parser.add_argument("--final-bucket-blend", type=float, default=0.0)
    parser.add_argument("--final-proba-power", type=float, default=0.0)
    parser.add_argument(
        "--prediction-strategy",
        default="catboost",
        choices=[
            "catboost",
            "pair_last_30d",
            "pair_last_90d_daily",
            "pair_last_180d_daily",
            "pair_blend_recent",
            "item_share_90d",
            "item_share_smooth",
            "conservative_min",
        ],
    )
    parser.add_argument("--baseline-scale-grid", type=float, nargs="+", default=[0.45, 0.55, 0.65, 0.75, 0.85, 0.95, 1.05, 1.15])
    return parser.parse_args()


def main():
    args = parse_args()
    ensure_output_dirs()
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    validate_inputs(args)

    bad_days = set(detect_bad_days(args.bad_day_mode))
    locations = candidate_locations()
    active_items = active_items_with_metadata()
    log(f"Bad days: {[d.isoformat() for d in sorted(bad_days)]}")
    log(f"Locations: {locations.height:,}; active items: {active_items.height:,}; candidates: {locations.height * active_items.height:,}")

    original_end_month = args.end_month
    tune = {
        "score": None,
        "threshold": args.final_threshold,
        "scale": args.final_scale,
        "blend": args.final_blend,
        "bucket_blend": args.final_bucket_blend,
        "proba_power": args.final_proba_power,
    }

    if args.mode in {"full", "validate-only"}:
        validation_months = sorted(args.validation_months or [args.validation_month])
        first_validation_month = validation_months[0]
        if first_validation_month <= args.start_month:
            raise ValueError("first validation month must be greater than --start-month")

        args.end_month = first_validation_month - 1
        train_df = build_training_data(args, bad_days, locations, active_items)
        log(f"Validation-training rows: {len(train_df):,}; columns: {len(train_df.columns):,}")
        clf, reg, bucket_model, bucket_values, feature_columns, cat_features = train_models(train_df, args)
        tune = tune_on_validation(args, bad_days, active_items, clf, reg, bucket_model, bucket_values, feature_columns, cat_features)
        if args.mode == "validate-only":
            return

    args.end_month = original_end_month
    final_train_df = build_training_data(args, bad_days, locations, active_items)
    log(f"Final-training rows: {len(final_train_df):,}; columns: {len(final_train_df.columns):,}")
    clf, reg, bucket_model, bucket_values, feature_columns, cat_features = train_models(final_train_df, args)

    clf.save_model(MODEL_DIR / "catboost_positive_classifier.cbm")
    reg.save_model(MODEL_DIR / "catboost_positive_quantity_regressor.cbm")
    if bucket_model is not None:
        bucket_model.save_model(MODEL_DIR / "catboost_quantity_bucket_classifier.cbm")
    metadata = {
        "bad_days": [d.isoformat() for d in sorted(bad_days)],
        "feature_columns": feature_columns,
        "cat_features": cat_features,
        "bucket_values": bucket_values.tolist() if bucket_values is not None else None,
        "tuning": tune,
        "args": vars(args),
    }
    (MODEL_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

    submission = final_prediction(
        args,
        bad_days,
        locations,
        active_items,
        clf,
        reg,
        bucket_model,
        bucket_values,
        feature_columns,
        cat_features,
        threshold=tune["threshold"],
        scale=tune["scale"],
        blend=tune["blend"],
        bucket_blend=tune["bucket_blend"],
        proba_power=tune["proba_power"],
    )
    log(f"Wrote {SUBMISSION_PATH} with {len(submission):,} rows")
    log(str(submission.dtypes))


if __name__ == "__main__":
    main()

Writing train_predict.py


### `train_predict_decval.py`

In [3]:
%%writefile train_predict_decval.py
import argparse
import json
import pickle
import time
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl

import train_predict as tp
from project_paths import OUTPUT_DIR, ensure_output_dirs


def log(message: str) -> None:
    print(message, flush=True)


def run_id() -> str:
    return time.strftime("decval_%Y%m%d_%H%M%S")


def month_days(year: int, month: int) -> int:
    start, end = tp.month_bounds(year, month)
    return (end - start).days + 1


def candidate_pairs_for_locations(locations: pl.DataFrame, active_items: pl.DataFrame, start: int, size: int) -> pl.DataFrame:
    loc_values = locations["location"].to_list()
    if size:
        loc_values = loc_values[:size]
    batch_locations = pl.DataFrame({"location": loc_values[start : start + size] if size else loc_values[start:]})
    return batch_locations.join(active_items.select("item_id"), how="cross")


def predict_candidate_grid(
    args,
    bad_days: set[date],
    locations: pl.DataFrame,
    active_items: pl.DataFrame,
    model_bundle: dict,
    cutoff: date,
    target_month: int,
    target_days: int,
    output_path: Path,
) -> pd.DataFrame:
    daily_lf = tp.daily_valid_lf(bad_days)
    feature_tables = tp.build_feature_tables(daily_lf, active_items, cutoff)
    rows: list[pd.DataFrame] = []
    loc_values = locations["location"].to_list()
    if args.max_prediction_locations:
        loc_values = loc_values[: args.max_prediction_locations]

    for start in range(0, len(loc_values), args.location_batch_size):
        batch_locations = pl.DataFrame({"location": loc_values[start : start + args.location_batch_size]})
        pairs = batch_locations.join(active_items.select("item_id"), how="cross")
        features = tp.add_features_from_tables(pairs, active_items, feature_tables, cutoff, target_month, target_days)
        if args.prediction_strategy == "catboost":
            pred = tp.predict_quantity(
                args,
                model_bundle["clf"],
                model_bundle["reg"],
                model_bundle["bucket_model"],
                model_bundle["bucket_values"],
                features,
                model_bundle["feature_columns"],
                model_bundle["cat_features"],
                model_bundle["tune"]["threshold"],
                model_bundle["tune"]["scale"],
                model_bundle["tune"]["blend"],
                model_bundle["tune"]["bucket_blend"],
                model_bundle["tune"]["proba_power"],
            )
        else:
            pred = tp.apply_baseline_strategy(features, args.prediction_strategy, args.final_scale)

        out = features[["location", "item_id"]].copy()
        out["quantity"] = pred.astype(np.int64)
        rows.append(out)
        log(f"Predicted {target_month:02d}: locations {start + 1:,}-{min(start + args.location_batch_size, len(loc_values)):,} / {len(loc_values):,}")

    submission = pd.concat(rows, ignore_index=True)
    submission = submission[["location", "item_id", "quantity"]]
    submission = pd.DataFrame(
        {
            "location": submission["location"].astype(str).to_numpy(dtype=object),
            "item_id": submission["item_id"].astype(str).to_numpy(dtype=object),
            "quantity": submission["quantity"].astype(np.int64).to_numpy(),
        }
    )
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("wb") as f:
        pickle.dump(submission, f)
    return submission


def score_december_submission(submission: pd.DataFrame, bad_days: set[date], active_items: pl.DataFrame) -> dict:
    daily_lf = tp.daily_valid_lf(bad_days)
    actual = tp.monthly_target(daily_lf, active_items, 2025, 12, bad_days).to_pandas()
    actual["location"] = actual["location"].astype(str)
    actual["item_id"] = actual["item_id"].astype(str)
    merged = actual.merge(submission, on=["location", "item_id"], how="left", suffixes=("_actual", "_pred"))
    merged["quantity_pred"] = merged["quantity_pred"].fillna(0)
    score = tp.score_metric(merged["quantity_actual"].to_numpy(), merged["quantity_pred"].to_numpy())
    ape = np.abs(merged["quantity_actual"].to_numpy() - merged["quantity_pred"].to_numpy()) / np.abs(
        merged["quantity_actual"].to_numpy()
    )
    return {
        "december_score": score,
        "december_positive_rows": int(len(merged)),
        "december_actual_sum": float(merged["quantity_actual"].sum()),
        "december_pred_sum_on_positive_rows": float(merged["quantity_pred"].sum()),
        "december_mean_ape": float(ape.mean()),
        "december_median_ape": float(np.median(ape)),
    }


def train_bundle(args, bad_days: set[date], locations: pl.DataFrame, active_items: pl.DataFrame, end_month: int) -> dict:
    original_end_month = args.end_month
    args.end_month = end_month
    train_df = tp.build_training_data(args, bad_days, locations, active_items)
    log(f"Training rows through month {end_month:02d}: {len(train_df):,}; columns: {len(train_df.columns):,}")
    clf, reg, bucket_model, bucket_values, feature_columns, cat_features = tp.train_models(train_df, args)
    args.end_month = original_end_month
    return {
        "clf": clf,
        "reg": reg,
        "bucket_model": bucket_model,
        "bucket_values": bucket_values,
        "feature_columns": feature_columns,
        "cat_features": cat_features,
    }


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--run-name", default="")
    parser.add_argument("--runs-dir", default=str(OUTPUT_DIR / "runs"))
    parser.add_argument("--bad-day-mode", choices=["threshold", "known", "none"], default="threshold")
    parser.add_argument("--start-month", type=int, default=4)
    parser.add_argument("--end-month", type=int, default=12)
    parser.add_argument("--validation-month", type=int, default=12)
    parser.add_argument("--validation-months", type=int, nargs="+", default=[12])
    parser.add_argument("--max-positive-per-month", type=int, default=120_000)
    parser.add_argument("--max-negative-per-month", type=int, default=240_000)
    parser.add_argument("--negative-ratio", type=int, default=2)
    parser.add_argument("--validation-positive-rows", type=int, default=300_000)
    parser.add_argument("--classifier-iterations", type=int, default=700)
    parser.add_argument("--regressor-iterations", type=int, default=900)
    parser.add_argument("--learning-rate", type=float, default=0.06)
    parser.add_argument("--depth", type=int, default=8)
    parser.add_argument("--regressor-loss", default="MAPE")
    parser.add_argument("--regressor-target", choices=["log", "raw"], default="raw")
    parser.add_argument("--use-bucket-model", action=argparse.BooleanOptionalAction, default=False)
    parser.add_argument("--bucket-iterations", type=int, default=500)
    parser.add_argument("--mape-weight-power", type=float, default=0.0)
    parser.add_argument("--mape-weight-floor", type=float, default=1.0)
    parser.add_argument("--classifier-positive-weight", type=float, default=1.0)
    parser.add_argument("--thread-count", type=int, default=-1)
    parser.add_argument("--task-type", choices=["CPU", "GPU"], default="CPU")
    parser.add_argument("--devices", default="0")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--location-batch-size", type=int, default=40)
    parser.add_argument("--max-prediction-locations", type=int, default=0)
    parser.add_argument("--threshold-grid", type=float, nargs="+", default=[0.0])
    parser.add_argument("--scale-grid", type=float, nargs="+", default=[0.5, 0.65, 0.75, 0.85, 1.0])
    parser.add_argument("--blend-grid", type=float, nargs="+", default=[0.0])
    parser.add_argument("--bucket-blend-grid", type=float, nargs="+", default=[0.0])
    parser.add_argument("--proba-power-grid", type=float, nargs="+", default=[0.0, 0.25, 0.5, 1.0])
    parser.add_argument("--final-threshold", type=float, default=0.0)
    parser.add_argument("--final-scale", type=float, default=0.75)
    parser.add_argument("--final-blend", type=float, default=0.0)
    parser.add_argument("--final-bucket-blend", type=float, default=0.0)
    parser.add_argument("--final-proba-power", type=float, default=0.0)
    parser.add_argument(
        "--prediction-strategy",
        default="catboost",
        choices=[
            "catboost",
            "pair_last_30d",
            "pair_last_90d_daily",
            "pair_last_180d_daily",
            "pair_blend_recent",
            "item_share_90d",
            "item_share_smooth",
            "conservative_min",
        ],
    )
    parser.add_argument("--baseline-scale-grid", type=float, nargs="+", default=[0.45, 0.55, 0.65, 0.75, 0.85, 0.95, 1.05, 1.15])
    return parser.parse_args()


def main():
    args = parse_args()
    ensure_output_dirs()
    tp.validate_inputs(args)

    run_root = Path(args.runs_dir) / (args.run_name or run_id())
    december_dir = run_root / "december_2025"
    january_dir = run_root / "january_2026"
    models_dir = run_root / "models"
    for path in [december_dir, january_dir, models_dir]:
        path.mkdir(parents=True, exist_ok=True)
    tp.MODEL_DIR = models_dir

    bad_days = set(tp.detect_bad_days(args.bad_day_mode))
    locations = tp.candidate_locations()
    active_items = tp.active_items_with_metadata()
    log(f"Run folder: {run_root}")
    log(f"Bad days: {[d.isoformat() for d in sorted(bad_days)]}")
    log(f"Locations: {locations.height:,}; active items: {active_items.height:,}; candidates: {locations.height * active_items.height:,}")

    # Train only on months before December, tune on December positives, and write a full-grid December submission.
    validation_bundle = train_bundle(args, bad_days, locations, active_items, end_month=11)
    tune = tp.tune_on_validation(
        args,
        bad_days,
        active_items,
        validation_bundle["clf"],
        validation_bundle["reg"],
        validation_bundle["bucket_model"],
        validation_bundle["bucket_values"],
        validation_bundle["feature_columns"],
        validation_bundle["cat_features"],
    )
    validation_bundle["tune"] = tune
    dec_submission = predict_candidate_grid(
        args,
        bad_days,
        locations,
        active_items,
        validation_bundle,
        cutoff=date(2025, 11, 30),
        target_month=12,
        target_days=31,
        output_path=december_dir / "submission.pkl",
    )
    metrics = score_december_submission(dec_submission, bad_days, active_items)

    # Retrain through December and predict January 2026 using the December-tuned parameters.
    final_bundle = train_bundle(args, bad_days, locations, active_items, end_month=12)
    final_bundle["tune"] = tune
    jan_submission = predict_candidate_grid(
        args,
        bad_days,
        locations,
        active_items,
        final_bundle,
        cutoff=date(2025, 12, 31),
        target_month=1,
        target_days=31,
        output_path=january_dir / "submission.pkl",
    )

    validation_bundle["clf"].save_model(models_dir / "validation_positive_classifier.cbm")
    validation_bundle["reg"].save_model(models_dir / "validation_quantity_regressor.cbm")
    final_bundle["clf"].save_model(models_dir / "final_positive_classifier.cbm")
    final_bundle["reg"].save_model(models_dir / "final_quantity_regressor.cbm")

    metrics.update(
        {
            "run_root": str(run_root),
            "december_submission": str(december_dir / "submission.pkl"),
            "january_submission": str(january_dir / "submission.pkl"),
            "january_rows": int(len(jan_submission)),
            "january_nonzero_rows": int((jan_submission["quantity"] > 0).sum()),
            "january_pred_sum": int(jan_submission["quantity"].sum()),
            "tuning": tune,
            "args": vars(args),
        }
    )
    (run_root / "metrics.txt").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    log(json.dumps(metrics, indent=2))


if __name__ == "__main__":
    main()

Writing train_predict_decval.py


### `train_predict_decval_v2.py`

In [4]:
%%writefile train_predict_decval_v2.py
import sys

from train_predict_decval import main


DEFAULT_ARGS = [
    "--run-name",
    "decval_v2",
    "--validation-months",
    "12",
    "--regressor-target",
    "raw",
    "--regressor-loss",
    "MAPE",
    "--no-use-bucket-model",
    "--threshold-grid",
    "0.0",
    "--scale-grid",
    "0.52",
    "0.55",
    "0.58",
    "0.61",
    "0.64",
    "0.67",
    "--blend-grid",
    "0.0",
    "--bucket-blend-grid",
    "0.0",
    "--proba-power-grid",
    "0.0",
    "--final-threshold",
    "0.0",
    "--final-scale",
    "0.67",
    "--final-blend",
    "0.0",
    "--final-bucket-blend",
    "0.0",
    "--final-proba-power",
    "0.0",
]


def has_flag(flag: str) -> bool:
    return flag in sys.argv[1:]


if __name__ == "__main__":
    # Keep user-provided flags authoritative. These defaults define the v2 run.
    injected = []
    i = 0
    while i < len(DEFAULT_ARGS):
        flag = DEFAULT_ARGS[i]
        values = []
        i += 1
        while i < len(DEFAULT_ARGS) and not DEFAULT_ARGS[i].startswith("--"):
            values.append(DEFAULT_ARGS[i])
            i += 1
        if not has_flag(flag):
            injected.extend([flag, *values])
    sys.argv = [sys.argv[0], *injected, *sys.argv[1:]]
    main()

Writing train_predict_decval_v2.py


### `prepare_cache.py`

In [5]:
%%writefile prepare_cache.py
from pathlib import Path

import polars as pl

from project_paths import CACHE_DIR, LOG_DIR, RAW_DATA_DIR, ensure_output_dirs

TRANSACTIONS_PATH = RAW_DATA_DIR / "transaction_full_2025.parquet"
CACHE_PATH = CACHE_DIR / "daily_quantity.parquet"
LOG_PATH = LOG_DIR / "prepare_quantity_cache_log.txt"


class Logger:
    def __init__(self, path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.file = self.path.open("w", encoding="utf-8")

    def write(self, message):
        print(message, flush=True)
        self.file.write(message + "\n")
        self.file.flush()

    def close(self):
        self.file.close()


def main():
    ensure_output_dirs()
    logger = Logger(LOG_PATH)
    try:
        cache_path = Path(CACHE_PATH)
        cache_path.parent.mkdir(parents=True, exist_ok=True)

        logger.write("Task: build daily quantity cache with Polars streaming")
        logger.write(f"Input: {TRANSACTIONS_PATH}")
        logger.write(f"Output: {CACHE_PATH}")
        logger.write("Streaming aggregate started...")

        (
            pl.scan_parquet(TRANSACTIONS_PATH)
            .filter(pl.col("event_type") == "Purchase")
            .select(
                "location",
                "item_id",
                pl.col("updated_date").dt.date().alias("date"),
                pl.col("quantity").cast(pl.Float32).alias("quantity"),
            )
            .group_by(["location", "item_id", "date"])
            .agg(pl.col("quantity").sum().alias("y"))
            .sink_parquet(CACHE_PATH)
        )

        logger.write(f"Saved daily quantity cache to {CACHE_PATH}")
        logger.write(f"Saved prepare log to {LOG_PATH}")
    finally:
        logger.close()


if __name__ == "__main__":
    main()

Writing prepare_cache.py


### `make_subset.py`

In [6]:
%%writefile make_subset.py
import argparse
from pathlib import Path

import polars as pl

from project_paths import CACHE_DIR, RAW_DATA_DIR


def log(message: str) -> None:
    print(message, flush=True)


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--output-dir", default="subset-data")
    parser.add_argument("--locations", type=int, default=40)
    parser.add_argument("--items", type=int, default=300)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--include-event", action="store_true")
    return parser.parse_args()


def main():
    args = parse_args()
    output_dir = Path(args.output_dir)
    raw_out = output_dir / "raw-data"
    cache_out = output_dir / "outputs" / "cache"
    raw_out.mkdir(parents=True, exist_ok=True)
    cache_out.mkdir(parents=True, exist_ok=True)

    items_path = RAW_DATA_DIR / "items.parquet"
    tx_path = RAW_DATA_DIR / "transaction_full_2025.parquet"
    daily_path = CACHE_DIR / "daily_quantity.parquet"

    active_items = (
        pl.scan_parquet(items_path)
        .filter(pl.col("sale_status") == 1)
        .select(pl.col("item_id").cast(pl.Utf8))
        .unique()
        .collect()
        .sample(n=args.items, seed=args.seed, shuffle=True)
    )
    locations = (
        pl.scan_parquet(tx_path)
        .select("location")
        .unique()
        .collect()
        .sample(n=args.locations, seed=args.seed + 1, shuffle=True)
    )

    log(f"Subset locations={locations.height:,}, active_items={active_items.height:,}")
    log(f"Candidate pairs={locations.height * active_items.height:,}")

    items_subset = (
        pl.scan_parquet(items_path)
        .with_columns(pl.col("item_id").cast(pl.Utf8))
        .join(active_items.lazy(), on="item_id", how="inner")
        .collect()
    )
    tx_subset = (
        pl.scan_parquet(tx_path)
        .with_columns(pl.col("item_id").cast(pl.Utf8))
        .join(active_items.lazy(), on="item_id", how="inner")
        .join(locations.lazy(), on="location", how="inner")
        .collect()
    )
    daily_subset = (
        pl.scan_parquet(daily_path)
        .with_columns(pl.col("item_id").cast(pl.Utf8))
        .join(active_items.lazy(), on="item_id", how="inner")
        .join(locations.lazy(), on="location", how="inner")
        .collect()
    )

    items_subset.write_parquet(raw_out / "items.parquet")
    tx_subset.write_parquet(raw_out / "transaction_full_2025.parquet")
    daily_subset.write_parquet(cache_out / "daily_quantity.parquet")

    if args.include_event:
        event_path = RAW_DATA_DIR / "event_full_2025.parquet"
        event_subset = (
            pl.scan_parquet(event_path)
            .with_columns(pl.col("item_id").cast(pl.Utf8))
            .join(active_items.lazy(), on="item_id", how="inner")
            .collect()
        )
        event_subset.write_parquet(raw_out / "event_full_2025.parquet")

    log(f"Wrote {raw_out / 'items.parquet'} rows={items_subset.height:,}")
    log(f"Wrote {raw_out / 'transaction_full_2025.parquet'} rows={tx_subset.height:,}")
    log(f"Wrote {cache_out / 'daily_quantity.parquet'} rows={daily_subset.height:,}")
    log("")
    log("Run local subset test:")
    log(f"  cd {output_dir}")
    log("  cp ../train_predict.py ../project_paths.py ../requirements.txt .")
    log("  python3 train_predict.py --mode full --start-month 10 --end-month 11 --validation-months 11 --max-positive-per-month 2000 --max-negative-per-month 4000 --validation-positive-rows 2000 --classifier-iterations 20 --regressor-iterations 20 --location-batch-size 10")


if __name__ == "__main__":
    main()

Writing make_subset.py


### `evaluator.py`

In [7]:
%%writefile evaluator.py
def evaluate(submission_path, answer_path):

    import pickle
    import pandas as pd
    import numpy as np

    # =====================================================
    # LOAD
    # =====================================================

    with open(submission_path, "rb") as f:
        submission = pickle.load(f)

    with open(answer_path, "rb") as f:
        answer = pickle.load(f)

    # =====================================================
    # CONVERT TO DATAFRAME
    # =====================================================

    submission_df = pd.DataFrame(submission)

    answer_df = pd.DataFrame(answer)

    submission_df["item_id"] = (
        submission_df["item_id"]
        .astype(str)
    )

    answer_df["item_id"] = (
        answer_df["item_id"]
        .astype(str)
    )

    # =====================================================
    # MERGE
    # =====================================================

    merged_df = answer_df.merge(
        submission_df,
        on=["location", "item_id"],
        how="left",
        suffixes=("_gt", "_pred")
    )

    # missing prediction -> 0
    merged_df["quantity_pred"] = (
        merged_df["quantity_pred"]
        .fillna(0)
    )

    # =====================================================
    # REMOVE ZERO GT
    # =====================================================

    merged_df = merged_df[
        merged_df["quantity_gt"] != 0
    ]

    if len(merged_df) == 0:
        return 0.0

    # =====================================================
    # MAPE
    # =====================================================

    ape = (
        np.abs(
            merged_df["quantity_gt"]
            - merged_df["quantity_pred"]
        )
        /
        np.abs(
            merged_df["quantity_gt"]
        )
    )

    mape = 100.0 - ape.mean() * 100

    return round(mape, 4)

Writing evaluator.py


### `fix_submission_pickle.py`

In [8]:
%%writefile fix_submission_pickle.py
import argparse
import pickle
from pathlib import Path

import numpy as np
import pandas as pd


DEFAULT_INPUT = Path("outputs/runs/decval_v2_scaled_max5/january_2026/submission.pkl")
DEFAULT_OUTPUT = Path("submission.pkl")


def normalize_submission(input_path: Path, output_path: Path) -> None:
    with input_path.open("rb") as f:
        submission = pickle.load(f)

    df = pd.DataFrame(submission)
    missing = {"location", "item_id", "quantity"} - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    normalized = pd.DataFrame(
        {
            "location": pd.to_numeric(df["location"], downcast="integer").to_numpy(dtype=np.int32),
            "item_id": df["item_id"].astype(str).to_numpy(dtype=object),
            "quantity": df["quantity"].fillna(0).round().clip(lower=0).to_numpy(dtype=np.int64),
        }
    )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("wb") as f:
        pickle.dump(normalized, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"Wrote {output_path}")
    print(f"shape={normalized.shape}")
    print(normalized.dtypes)


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Rewrite submission.pkl to match evaluator dtypes.")
    parser.add_argument("--input", type=Path, default=DEFAULT_INPUT)
    parser.add_argument("--output", type=Path, default=DEFAULT_OUTPUT)
    parser.add_argument(
        "--all-runs",
        action="store_true",
        help="Overwrite every outputs/runs/*/*/submission.pkl in place.",
    )
    return parser.parse_args()


if __name__ == "__main__":
    args = parse_args()
    if args.all_runs:
        paths = sorted(Path("outputs/runs").glob("*/*/submission.pkl"))
        if not paths:
            raise SystemExit("No outputs/runs/*/*/submission.pkl files found.")
        for path in paths:
            normalize_submission(path, path)
    else:
        normalize_submission(args.input, args.output)

Writing fix_submission_pickle.py


### `eda.py`

In [9]:
%%writefile eda.py
from pathlib import Path
import argparse
import json

import matplotlib.pyplot as plt
import polars as pl

from project_paths import CACHE_DIR, OUTPUT_DIR, RAW_DATA_DIR, ensure_output_dirs, first_existing


CACHE_PATH = first_existing(CACHE_DIR / "daily_quantity.parquet", "outputs/daily_quantity.parquet")
TRANSACTIONS_PATH = RAW_DATA_DIR / "transaction_full_2025.parquet"
EDA_DIR = OUTPUT_DIR / "eda"


def log(message):
    print(message, flush=True)


def scan_daily_quantity():
    if Path(CACHE_PATH).exists():
        log(f"Reading daily cache: {CACHE_PATH}")
        return pl.scan_parquet(CACHE_PATH).select(
            pl.col("location"),
            pl.col("item_id").cast(pl.Utf8).alias("item_id"),
            pl.col("date").cast(pl.Date).alias("date"),
            pl.col("y").cast(pl.Float64).alias("quantity"),
        )

    if Path(TRANSACTIONS_PATH).exists():
        log(f"Daily cache not found. Reading raw transactions: {TRANSACTIONS_PATH}")
        return (
            pl.scan_parquet(TRANSACTIONS_PATH)
            .filter(pl.col("event_type") == "Purchase")
            .select(
                "location",
                pl.col("item_id").cast(pl.Utf8).alias("item_id"),
                pl.col("updated_date").dt.date().alias("date"),
                pl.col("quantity").cast(pl.Float64).alias("quantity"),
            )
            .group_by(["location", "item_id", "date"])
            .agg(pl.col("quantity").sum().alias("quantity"))
        )

    raise FileNotFoundError(
        f"Missing both cache {CACHE_PATH} and raw transaction file {TRANSACTIONS_PATH}"
    )


def add_date_features(lf):
    return lf.with_columns(
        pl.col("date").dt.weekday().alias("weekday"),
        pl.col("date").dt.month().alias("month"),
        pl.col("date").dt.ordinal_day().alias("day_of_year"),
    )


def dataset_overview(daily_lf):
    overview = daily_lf.select(
        pl.col("date").min().alias("min_date"),
        pl.col("date").max().alias("max_date"),
        pl.len().alias("nonzero_pair_days"),
        pl.struct(["location", "item_id"]).n_unique().alias("pairs"),
        pl.col("location").n_unique().alias("locations"),
        pl.col("item_id").n_unique().alias("items"),
        pl.col("quantity").sum().alias("total_quantity"),
        pl.col("quantity").mean().alias("mean_quantity_on_sold_day"),
        pl.col("quantity").median().alias("median_quantity_on_sold_day"),
        pl.col("quantity").quantile(0.90).alias("p90_quantity_on_sold_day"),
        pl.col("quantity").quantile(0.99).alias("p99_quantity_on_sold_day"),
        pl.col("quantity").max().alias("max_quantity_on_sold_day"),
    ).collect()
    min_date, max_date = overview.select("min_date", "max_date").row(0)
    n_days = (max_date - min_date).days + 1
    return overview.with_columns(
        pl.lit(n_days).alias("calendar_days"),
        (pl.col("nonzero_pair_days") / (pl.col("pairs") * n_days)).alias("global_active_rate"),
    )


def pair_summary(daily_lf):
    base = (
        daily_lf.group_by(["location", "item_id"])
        .agg(
            pl.col("date").min().alias("first_sold_date"),
            pl.col("date").max().alias("last_sold_date"),
            pl.col("date").n_unique().alias("active_days"),
            pl.col("quantity").sum().alias("total_quantity"),
            pl.col("quantity").mean().alias("mean_quantity_on_sold_day"),
            pl.col("quantity").median().alias("median_quantity_on_sold_day"),
            pl.col("quantity").std().fill_null(0).alias("std_quantity_on_sold_day"),
            pl.col("quantity").quantile(0.25).alias("q25_quantity_on_sold_day"),
            pl.col("quantity").quantile(0.75).alias("q75_quantity_on_sold_day"),
            pl.col("quantity").quantile(0.90).alias("p90_quantity_on_sold_day"),
            pl.col("quantity").quantile(0.99).alias("p99_quantity_on_sold_day"),
            pl.col("quantity").max().alias("max_quantity_on_sold_day"),
        )
        .with_columns(
            (
                (pl.col("last_sold_date") - pl.col("first_sold_date")).dt.total_days() + 1
            ).alias("observed_span_days"),
        )
        .with_columns(
            (pl.col("active_days") / pl.col("observed_span_days")).alias("active_rate_in_span"),
            (
                pl.col("std_quantity_on_sold_day")
                / pl.col("mean_quantity_on_sold_day").clip(lower_bound=1e-9)
            ).alias("cv_quantity_on_sold_day"),
            (
                pl.col("max_quantity_on_sold_day")
                / pl.col("median_quantity_on_sold_day").clip(lower_bound=1e-9)
            ).alias("max_to_median_ratio"),
        )
    )

    gaps = (
        daily_lf.sort(["location", "item_id", "date"])
        .with_columns(
            (
                pl.col("date").diff().over(["location", "item_id"]).dt.total_days()
            ).alias("gap_days")
        )
        .filter(pl.col("gap_days").is_not_null())
        .group_by(["location", "item_id"])
        .agg(
            pl.col("gap_days").mean().alias("mean_gap_between_sold_days"),
            pl.col("gap_days").median().alias("median_gap_between_sold_days"),
            pl.col("gap_days").max().alias("max_gap_between_sold_days"),
        )
    )

    return base.join(gaps, on=["location", "item_id"], how="left").with_columns(
        pl.col("mean_gap_between_sold_days").fill_null(0),
        pl.col("median_gap_between_sold_days").fill_null(0),
        pl.col("max_gap_between_sold_days").fill_null(0),
    )


def outlier_days(daily_lf, pair_stats_lf):
    thresholds = pair_stats_lf.select(
        "location",
        "item_id",
        (
            pl.col("q75_quantity_on_sold_day")
            + 1.5 * (pl.col("q75_quantity_on_sold_day") - pl.col("q25_quantity_on_sold_day"))
        ).alias("high_outlier_threshold"),
    )
    return (
        daily_lf.join(thresholds, on=["location", "item_id"], how="inner")
        .filter(pl.col("quantity") > pl.col("high_outlier_threshold"))
        .select(["location", "item_id", "date", "quantity", "high_outlier_threshold"])
        .sort("quantity", descending=True)
    )


def write_table(df_or_lf, path, top_n=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df = df_or_lf.collect(streaming=True) if isinstance(df_or_lf, pl.LazyFrame) else df_or_lf
    if top_n is not None:
        df = df.head(top_n)
    df.write_csv(path)
    log(f"Wrote {path} ({df.height:,} rows)")
    return df


def write_json_summary(overview_df, path):
    row = overview_df.to_dicts()[0]
    serializable = {key: str(value) if hasattr(value, "isoformat") else value for key, value in row.items()}
    Path(path).write_text(json.dumps(serializable, indent=2), encoding="utf-8")
    log(f"Wrote {path}")


def save_bar(df, x, y, title, path, rotation=45):
    if df.height == 0:
        return
    plt.figure(figsize=(12, 6))
    plt.bar(df[x].cast(str).to_list(), df[y].to_list())
    plt.title(title)
    plt.xlabel(x)
    plt.ylabel(y)
    plt.xticks(rotation=rotation, ha="right")
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()
    log(f"Wrote {path}")


def save_line(df, x, y, title, path):
    if df.height == 0:
        return
    plt.figure(figsize=(14, 6))
    plt.plot(df[x].to_list(), df[y].to_list(), linewidth=1.2)
    plt.title(title)
    plt.xlabel(x)
    plt.ylabel(y)
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()
    log(f"Wrote {path}")


def save_hist(values, title, path, bins=80, log_y=True):
    values = [v for v in values if v is not None]
    if not values:
        return
    plt.figure(figsize=(12, 6))
    plt.hist(values, bins=bins)
    if log_y:
        plt.yscale("log")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()
    log(f"Wrote {path}")


def save_scatter(df, x, y, title, path):
    if df.height == 0:
        return
    sample = df.head(50000)
    plt.figure(figsize=(10, 7))
    plt.scatter(sample[x].to_list(), sample[y].to_list(), s=4, alpha=0.25)
    plt.xscale("log")
    plt.yscale("log")
    plt.title(title)
    plt.xlabel(x)
    plt.ylabel(y)
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()
    log(f"Wrote {path}")


def generate_plots(daily_lf, pair_stats_df):
    daily_totals = (
        daily_lf.group_by("date")
        .agg(
            pl.col("quantity").sum().alias("total_quantity"),
            pl.len().alias("sold_pair_days"),
        )
        .sort("date")
        .collect(streaming=True)
    )
    save_line(
        daily_totals,
        "date",
        "total_quantity",
        "Total sold quantity by day, excluding zero pair-days",
        EDA_DIR / "daily_total_quantity.png",
    )
    save_line(
        daily_totals,
        "date",
        "sold_pair_days",
        "Number of active location-item pairs by day",
        EDA_DIR / "daily_active_pairs.png",
    )

    weekday = (
        add_date_features(daily_lf)
        .group_by("weekday")
        .agg(pl.col("quantity").sum().alias("total_quantity"))
        .sort("weekday")
        .collect(streaming=True)
    )
    save_bar(weekday, "weekday", "total_quantity", "Total quantity by weekday", EDA_DIR / "weekday_quantity.png", 0)

    top_locations = (
        daily_lf.group_by("location")
        .agg(pl.col("quantity").sum().alias("total_quantity"))
        .sort("total_quantity", descending=True)
        .limit(30)
        .collect(streaming=True)
    )
    save_bar(
        top_locations,
        "location",
        "total_quantity",
        "Top 30 locations by sold quantity",
        EDA_DIR / "top_locations_quantity.png",
    )

    top_items = (
        daily_lf.group_by("item_id")
        .agg(pl.col("quantity").sum().alias("total_quantity"))
        .sort("total_quantity", descending=True)
        .limit(30)
        .collect(streaming=True)
    )
    save_bar(top_items, "item_id", "total_quantity", "Top 30 items by sold quantity", EDA_DIR / "top_items_quantity.png")

    quantity_sample = daily_lf.select("quantity").limit(2_000_000).collect(streaming=True)
    save_hist(
        quantity_sample["quantity"].to_list(),
        "Sold-day quantity distribution, zeros excluded",
        EDA_DIR / "sold_day_quantity_hist.png",
    )
    save_hist(
        pair_stats_df["active_days"].to_list(),
        "Active sold-day count per location-item pair",
        EDA_DIR / "pair_active_days_hist.png",
    )
    save_scatter(
        pair_stats_df.sort("total_quantity", descending=True),
        "active_days",
        "total_quantity",
        "Pair active days vs total quantity, zeros excluded",
        EDA_DIR / "pair_active_days_vs_total_quantity.png",
    )


def focus_pair(daily_lf, location, item_id):
    pair_df = (
        daily_lf.filter((pl.col("location") == location) & (pl.col("item_id") == str(item_id)))
        .sort("date")
        .collect(streaming=True)
    )
    if pair_df.height == 0:
        log(f"No non-zero daily sales found for location={location}, item_id={item_id}")
        return

    pair_path = EDA_DIR / f"pair_location_{location}_item_{item_id}.csv"
    pair_df.write_csv(pair_path)
    log(f"Wrote {pair_path} ({pair_df.height:,} rows)")

    save_line(
        pair_df,
        "date",
        "quantity",
        f"Daily quantity for location={location}, item_id={item_id}; zeros excluded",
        EDA_DIR / f"pair_location_{location}_item_{item_id}.png",
    )


def main():
    parser = argparse.ArgumentParser(description="EDA for daily quantity by (location, item_id).")
    parser.add_argument("--location", type=int, default=None, help="Optional location to inspect.")
    parser.add_argument("--item-id", default=None, help="Optional item_id to inspect.")
    parser.add_argument("--top-n", type=int, default=1000, help="Rows to export for ranked CSV tables.")
    args = parser.parse_args()

    ensure_output_dirs()
    EDA_DIR.mkdir(parents=True, exist_ok=True)

    daily_lf = scan_daily_quantity().filter(pl.col("quantity") > 0)
    overview = dataset_overview(daily_lf)
    write_table(overview, EDA_DIR / "overview.csv")
    write_json_summary(overview, EDA_DIR / "overview.json")

    pair_stats_lf = pair_summary(daily_lf)
    pair_stats_df = pair_stats_lf.collect(streaming=True)
    pair_stats_df.write_parquet(EDA_DIR / "pair_summary.parquet")
    log(f"Wrote {EDA_DIR / 'pair_summary.parquet'} ({pair_stats_df.height:,} rows)")

    write_table(
        pair_stats_df.sort("total_quantity", descending=True),
        EDA_DIR / "top_pairs_by_total_quantity.csv",
        args.top_n,
    )
    write_table(
        pair_stats_df.sort("active_days", descending=True),
        EDA_DIR / "top_pairs_by_active_days.csv",
        args.top_n,
    )
    write_table(
        pair_stats_df.filter(pl.col("active_days") >= 10).sort("cv_quantity_on_sold_day", descending=True),
        EDA_DIR / "volatile_pairs_min_10_active_days.csv",
        args.top_n,
    )
    write_table(
        pair_stats_df.sort("max_gap_between_sold_days", descending=True),
        EDA_DIR / "intermittent_pairs_by_max_gap.csv",
        args.top_n,
    )
    write_table(
        outlier_days(daily_lf, pair_stats_lf),
        EDA_DIR / "high_outlier_sold_days.csv",
        args.top_n,
    )

    location_summary = (
        daily_lf.group_by("location")
        .agg(
            pl.col("quantity").sum().alias("total_quantity"),
            pl.len().alias("nonzero_pair_days"),
            pl.col("item_id").n_unique().alias("items_sold"),
            pl.struct(["location", "item_id"]).n_unique().alias("pairs_sold"),
        )
        .sort("total_quantity", descending=True)
    )
    write_table(location_summary, EDA_DIR / "location_summary.csv")

    item_summary = (
        daily_lf.group_by("item_id")
        .agg(
            pl.col("quantity").sum().alias("total_quantity"),
            pl.len().alias("nonzero_pair_days"),
            pl.col("location").n_unique().alias("locations_sold"),
            pl.struct(["location", "item_id"]).n_unique().alias("pairs_sold"),
        )
        .sort("total_quantity", descending=True)
    )
    write_table(item_summary, EDA_DIR / "item_summary.csv")

    date_summary = (
        add_date_features(daily_lf)
        .group_by("date")
        .agg(
            pl.col("quantity").sum().alias("total_quantity"),
            pl.len().alias("nonzero_pair_days"),
            pl.col("location").n_unique().alias("active_locations"),
            pl.col("item_id").n_unique().alias("active_items"),
        )
        .sort("date")
    )
    write_table(date_summary, EDA_DIR / "date_summary.csv")

    weekday_summary = (
        add_date_features(daily_lf)
        .group_by("weekday")
        .agg(
            pl.col("quantity").sum().alias("total_quantity"),
            pl.col("quantity").mean().alias("mean_quantity_on_sold_day"),
            pl.len().alias("nonzero_pair_days"),
        )
        .sort("weekday")
    )
    write_table(weekday_summary, EDA_DIR / "weekday_summary.csv")

    generate_plots(daily_lf, pair_stats_df)

    if args.location is not None and args.item_id is not None:
        focus_pair(daily_lf, args.location, args.item_id)
    elif args.location is not None or args.item_id is not None:
        log("Provide both --location and --item-id to export one pair's daily series.")

    log(f"EDA complete. Open files under {EDA_DIR}")


if __name__ == "__main__":
    main()

Writing eda.py


## Execute Pipeline
Run the cell below to execute the full validation and forecasting pipeline.

In [10]:
!python train_predict_decval_v2.py \
  --task-type CPU \
  --devices 0 \
  --start-month 4 \
  --end-month 12 \
  --max-positive-per-month 120000 \
  --max-negative-per-month 240000 \
  --validation-positive-rows 300000 \
  --classifier-iterations 700 \
  --regressor-iterations 900 \
  --location-batch-size 40

Missing /kaggle/working/cache/daily_quantity.parquet; building daily cache from raw transactions
Run folder: /kaggle/working/runs/decval_v2
Bad days: ['2025-01-29', '2025-01-30', '2025-01-31', '2025-07-04', '2025-07-18', '2025-09-29', '2025-10-27']
Locations: 1,038; active items: 6,850; candidates: 7,110,300
Month 04: cutoff=2025-03-31, positives=120,000, negatives=240,000, target_end=2025-04-30
START feature tables cutoff=2025-03-31
DONE feature tables cutoff=2025-03-31 in 10.4s
Month 05: cutoff=2025-04-30, positives=120,000, negatives=240,000, target_end=2025-05-31
START feature tables cutoff=2025-04-30
DONE feature tables cutoff=2025-04-30 in 13.8s
Month 06: cutoff=2025-05-31, positives=120,000, negatives=240,000, target_end=2025-06-30
START feature tables cutoff=2025-05-31
DONE feature tables cutoff=2025-05-31 in 17.4s
Month 07: cutoff=2025-06-30, positives=120,000, negatives=240,000, target_end=2025-07-31
START feature tables cutoff=2025-06-30
DONE feature tables cutoff=2025-06-30

In [11]:
!python fix_submission_pickle.py --input /kaggle/working/runs/decval_v2/january_2026/submission.pkl --output /kaggle/working/submission_fixed.pkl

Wrote /kaggle/working/submission_fixed.pkl
shape=(7110300, 3)
location     int32
item_id     object
quantity     int64
dtype: object
